In [ ]:
!nvidia-smi

In [ ]:
# TerraIncognita code-only downloader via DomainBed (no local dataset needed)
import os
import shutil
from pathlib import Path
import subprocess

os.chdir('/content')
domainbed_root = Path('/content/DomainBed')
data_root = Path('/content/DomainBed/domainbed/data/terra_incognita')

def _looks_like_terra_root(path_obj):
    if not path_obj.exists() or not path_obj.is_dir():
        return False
    domain_dirs = [d for d in path_obj.iterdir() if d.is_dir()]
    if len(domain_dirs) < 4:
        return False
    valid_domains = 0
    for dom in domain_dirs:
        class_dirs = [c for c in dom.iterdir() if c.is_dir()]
        if not class_dirs:
            continue
        has_image = False
        for cls in class_dirs:
            for fp in cls.rglob('*'):
                if fp.is_file() and fp.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}:
                    has_image = True
                    break
            if has_image:
                break
        if has_image:
            valid_domains += 1
    return valid_domains >= 4

def _find_valid_root(base_dir, max_depth=5):
    if not base_dir.exists() or not base_dir.is_dir():
        return None
    if _looks_like_terra_root(base_dir):
        return base_dir
    stack = [(base_dir, 0)]
    while stack:
        cur, depth = stack.pop()
        if depth >= max_depth:
            continue
        try:
            children = [p for p in cur.iterdir() if p.is_dir()]
        except Exception:
            continue
        for ch in children:
            if _looks_like_terra_root(ch):
                return ch
            stack.append((ch, depth + 1))
    return None

existing_root = _find_valid_root(data_root)
if existing_root is not None:
    print('TerraIncognita already present at:', existing_root)
else:
    if domainbed_root.exists():
        shutil.rmtree(domainbed_root)

    print('Cloning DomainBed...')
    subprocess.run(['git', 'clone', '-q', 'https://github.com/facebookresearch/DomainBed.git'], check=True)

    print('Installing missing dependency: wilds')
    subprocess.run(['pip', 'install', '-q', 'wilds'], check=True)

    print('Patching DomainBed download script to TerraIncognita only...')
    dl_path = Path('/content/DomainBed/domainbed/scripts/download.py')
    dl_content = dl_path.read_text(encoding='utf-8')
    dl_content = dl_content.replace('datasets.DATASETS', "['TerraIncognita']")
    dl_path.write_text(dl_content, encoding='utf-8')

    print('Downloading and formatting TerraIncognita (~6.5GB)...')
    subprocess.run(
        ['python', '-m', 'domainbed.scripts.download', '--data_dir=./domainbed/data'],
        cwd=str(domainbed_root),
        check=True,
    )
    print('Download and formatting complete.')

if not data_root.exists():
    raise FileNotFoundError(f'TerraIncognita folder not found at {data_root}')

resolved_root = _find_valid_root(data_root)
if resolved_root is None:
    raise ValueError(f'TerraIncognita exists at {data_root}, but no valid root/domain/class/images layout was found.')

envs = sorted([p.name for p in resolved_root.iterdir() if p.is_dir()])
print('Detected environments:', envs)
for env in envs:
    n_files = sum(1 for fp in (resolved_root / env).rglob('*') if fp.is_file())
    print(f'  {env}: {n_files} files')

# Pass root to next setup cell
MANUAL_TERRA_ROOT = str(resolved_root)
print('MANUAL_TERRA_ROOT set to:', MANUAL_TERRA_ROOT)
print('Now run Cell 3.')

In [ ]:
# ==========================================
# TerraIncognita: Setup + Upload Path + Real Dataset Stats
# ==========================================

%pip install -q timm torchvision seaborn

import gc
import math
import time
import random
import warnings
import os
import shutil
import subprocess
from pathlib import Path
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import timm
from PIL import Image
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Methods and hyperparameter policy:
# - Same methodology as CivilComments pipeline
# - Same method-specific robust hyperparameters
# - Adam optimizer for all methods (per your request)
RUN_METHODS = ["ERM", "SAM", "IRS-Instance", "CVaR-DRO"]
MODEL_NAME = "deit_small_patch16_224"

# Start small for smoke runs; set to 1.0 for full runs.
DATA_FRACTION = 1.0

# Suggested defaults for DeiT-style fine-tuning (not full 300-epoch pretraining).
SHARED_HPARAMS = {
    "train_batch_size": 256,
    "learning_rate": 2.5e-4,
    "num_train_epochs": 10,
    "weight_decay": 0.0,
    "num_workers": 0,
    "image_size": 224,
}

METHOD_HPARAMS = {
    "SAM": {"rho": 0.05},
    "CVaR-DRO": {"alpha": 0.1},
    "IRS-Instance": {
        "tau": 0.1,
        "distance_scale": 1.0,
        "min_div": 1e-2,
        "h_max": 100.0,
        "tau_eps": 1e-4,
        "tau_mult": 1.01,
        "warmup_epochs": 1,
    },
}

SEED = 42


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

print("Methods:", RUN_METHODS)
print("Model:", MODEL_NAME)
print("DATA_FRACTION:", DATA_FRACTION)
print("Shared hyperparams:", SHARED_HPARAMS)
print("Method hyperparams:", METHOD_HPARAMS)


# ---------- TerraIncognita local-data loader ----------
# Expected folder structure (example):
# /content/terra_incognita/<domain_name>/<class_name>/*.jpg
# where there are 4 domain folders.

TERRA_ROOT_CANDIDATES = [
    "/content/terra_incognita",
    "/content/TerraIncognita",
    "/content/DomainBed/domainbed/data/terra_incognita",
    "/content/drive/MyDrive/terra_incognita",
    "/content/drive/MyDrive/TerraIncognita",
]

ZIP_CANDIDATES = [
    "/content/terra_incognita.zip",
    "/content/TerraIncognita.zip",
    "/content/drive/MyDrive/terra_incognita.zip",
    "/content/drive/MyDrive/TerraIncognita.zip",
]

# Optional manual override. Set this if your extracted folder is custom.
MANUAL_TERRA_ROOT = globals().get("MANUAL_TERRA_ROOT", None)
DIRECT_UPLOAD_ENABLED = True  # Works in Colab without Drive.
DOWNLOAD_WITH_DOMAINBED_IF_MISSING = True


def is_image_file(path):
    return path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def find_terra_root(candidates):
    for c in candidates:
        p = Path(c)
        if p.exists() and p.is_dir() and looks_like_terra_root(p):
            return p
    return None


def looks_like_terra_root(path_obj):
    if not path_obj.exists() or not path_obj.is_dir():
        return False

    domain_dirs = [d for d in path_obj.iterdir() if d.is_dir()]
    if len(domain_dirs) < 4:
        return False

    valid_domains = 0
    for dom in domain_dirs:
        class_dirs = [c for c in dom.iterdir() if c.is_dir()]
        if not class_dirs:
            continue

        has_image = False
        for cls in class_dirs:
            for fp in cls.rglob("*"):
                if fp.is_file() and is_image_file(fp):
                    has_image = True
                    break
            if has_image:
                break

        if has_image:
            valid_domains += 1

    return valid_domains >= 4


def recursive_find_terra_root(search_roots, max_depth=5):
    for root in search_roots:
        rp = Path(root)
        if not rp.exists() or not rp.is_dir():
            continue

        if looks_like_terra_root(rp):
            return rp

        stack = [(rp, 0)]
        while stack:
            cur, depth = stack.pop()
            if depth >= max_depth:
                continue

            try:
                children = [p for p in cur.iterdir() if p.is_dir()]
            except Exception:
                continue

            for ch in children:
                if looks_like_terra_root(ch):
                    return ch
                stack.append((ch, depth + 1))

    return None


def try_extract_zip(zip_candidates):
    for z in zip_candidates:
        zp = Path(z)
        if zp.exists() and zp.is_file():
            print(f"Found zip: {zp}. Extracting to /content ...")
            with zipfile.ZipFile(zp, "r") as zf:
                zf.extractall("/content")
            return True
    return False


def try_colab_upload_zip(enabled=True):
    if not enabled:
        return False

    try:
        from google.colab import files
    except Exception:
        # Not in Colab (or upload API unavailable).
        return False

    print("No Terra folder found yet. Upload Terra zip directly from local machine (no Drive).")
    print("Pick one .zip file containing root/domain/class/images.")

    uploaded = files.upload()
    if not uploaded:
        return False

    zip_names = [name for name in uploaded.keys() if str(name).lower().endswith(".zip")]
    if not zip_names:
        print("Upload received, but no .zip file detected.")
        return False

    zip_path = Path("/content") / zip_names[0]
    print(f"Extracting uploaded zip: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall("/content")
    return True


def try_download_terra_with_domainbed(enabled=True):
    if not enabled:
        return False

    domainbed_root = Path("/content/DomainBed")
    target_root = Path("/content/DomainBed/domainbed/data/terra_incognita")
    if target_root.exists() and target_root.is_dir() and looks_like_terra_root(target_root):
        print("DomainBed TerraIncognita already exists.")
        return True

    print("Attempting TerraIncognita download via DomainBed...")
    try:
        os.chdir("/content")
        if domainbed_root.exists():
            shutil.rmtree(domainbed_root)

        subprocess.run(["git", "clone", "-q", "https://github.com/facebookresearch/DomainBed.git"], check=True)
        subprocess.run(["pip", "install", "-q", "wilds"], check=True)

        dl_path = Path("/content/DomainBed/domainbed/scripts/download.py")
        content = dl_path.read_text(encoding="utf-8")
        content = content.replace("datasets.DATASETS", "['TerraIncognita']")
        dl_path.write_text(content, encoding="utf-8")

        subprocess.run(["python", "-m", "domainbed.scripts.download", "--data_dir=./domainbed/data"], cwd=str(domainbed_root), check=True)

        valid = target_root.exists() and target_root.is_dir() and looks_like_terra_root(target_root)
        print("DomainBed download success:", valid)
        return valid
    except Exception as e:
        print("DomainBed download failed:", str(e))
        return False


def collect_terra_records(root_dir):
    domain_dirs = sorted([d for d in root_dir.iterdir() if d.is_dir()])
    if len(domain_dirs) < 2:
        raise ValueError(
            f"Found {len(domain_dirs)} domain folders in {root_dir}. "
            "Expected multi-domain folder layout like root/domain/class/images."
        )

    domain_names = [d.name for d in domain_dirs]
    domain_to_id = {name: i for i, name in enumerate(domain_names)}

    class_names_set = set()
    rows = []
    for dom in domain_dirs:
        class_dirs = sorted([c for c in dom.iterdir() if c.is_dir()])
        for cls in class_dirs:
            class_names_set.add(cls.name)

    class_names = sorted(class_names_set)
    class_to_id = {name: i for i, name in enumerate(class_names)}

    for dom in domain_dirs:
        dom_id = domain_to_id[dom.name]
        class_dirs = sorted([c for c in dom.iterdir() if c.is_dir()])
        for cls in class_dirs:
            cls_id = class_to_id[cls.name]
            for fp in cls.rglob("*"):
                if fp.is_file() and is_image_file(fp):
                    rows.append((str(fp), dom_id, cls_id, dom.name, cls.name))

    if len(rows) == 0:
        raise ValueError(
            f"No image files found under {root_dir}. "
            "Check upload path and folder structure root/domain/class/*.jpg"
        )

    df = pd.DataFrame(rows, columns=["filepath", "domain", "label", "domain_name", "class_name"])
    return df, domain_to_id, class_to_id


# 1) Manual override takes precedence.
terra_root = None
if MANUAL_TERRA_ROOT is not None and Path(MANUAL_TERRA_ROOT).exists():
    manual_root = Path(MANUAL_TERRA_ROOT)
    if looks_like_terra_root(manual_root):
        terra_root = manual_root
    else:
        terra_root = recursive_find_terra_root([str(manual_root)], max_depth=5)
        if terra_root is not None:
            print(f'MANUAL_TERRA_ROOT was not the exact root; using nested valid root: {terra_root}')
        else:
            print(f'Ignoring invalid MANUAL_TERRA_ROOT (no valid domain/class/images layout): {manual_root}')

# 2) Check known folders.
if terra_root is None:
    terra_root = find_terra_root(TERRA_ROOT_CANDIDATES)

# 3) Recursive search under /content and /content/drive/MyDrive.
if terra_root is None:
    terra_root = recursive_find_terra_root(["/content", "/content/drive/MyDrive"], max_depth=5)

# 4) Try extracting common zip names, then search again.
if terra_root is None:
    extracted = try_extract_zip(ZIP_CANDIDATES)
    if extracted:
        terra_root = find_terra_root(TERRA_ROOT_CANDIDATES)
        if terra_root is None:
            terra_root = recursive_find_terra_root(["/content", "/content/drive/MyDrive"], max_depth=5)

# 5) Optional direct upload (no Drive) when running in Colab.
if terra_root is None:
    uploaded = try_colab_upload_zip(enabled=DIRECT_UPLOAD_ENABLED)
    if uploaded:
        terra_root = find_terra_root(TERRA_ROOT_CANDIDATES)
        if terra_root is None:
            terra_root = recursive_find_terra_root(["/content", "/content/drive/MyDrive"], max_depth=5)

# 6) Download Terra with DomainBed (code-only path).
if terra_root is None:
    fetched = try_download_terra_with_domainbed(enabled=DOWNLOAD_WITH_DOMAINBED_IF_MISSING)
    if fetched:
        terra_root = find_terra_root(TERRA_ROOT_CANDIDATES)
        if terra_root is None:
            terra_root = recursive_find_terra_root(["/content", "/content/drive/MyDrive"], max_depth=5)

if terra_root is None:
    available_content = []
    p_content = Path("/content")
    if p_content.exists():
        available_content = sorted([p.name for p in p_content.iterdir()])

    raise FileNotFoundError(
        "TerraIncognita data folder not found. Upload/mount dataset first.\n"
        "Expected one of: " + ", ".join(TERRA_ROOT_CANDIDATES) + "\n"
        "Or set MANUAL_TERRA_ROOT to the correct extracted folder.\n"
        "Known zip candidates: " + ", ".join(ZIP_CANDIDATES) + "\n"
        "Current /content entries: " + ", ".join(available_content) + "\n\n"
        "No-Drive quick setup (Colab):\n"
        "1) from google.colab import files\n"
        "2) uploaded = files.upload()  # choose Terra zip from your computer\n"
        "3) !unzip -q /content/<uploaded_zip_name>.zip -d /content/\n"
        "4) Set MANUAL_TERRA_ROOT if needed, then rerun this cell.\n\n"
        "Colab quick setup:\n"
        "1) from google.colab import drive\n"
        "2) drive.mount('/content/drive')\n"
        "3) !unzip -q /content/drive/MyDrive/<your_terra_zip>.zip -d /content/\n"
        "4) Set MANUAL_TERRA_ROOT to your extracted folder path"
    )

print("Using TerraIncognita root:", terra_root)

stats_df, domain_to_id, class_to_id = collect_terra_records(terra_root)
NUM_CLASSES = len(class_to_id)

# Build maps to keep same variable contract as downstream cells.
DOMAIN_NAME_MAP = {v: k for k, v in domain_to_id.items()}
CLASS_NAME_MAP = {v: k for k, v in class_to_id.items()}
ALL_DOMAIN_IDS = sorted(DOMAIN_NAME_MAP.keys())

# Keep same record schema used later: (split_name, raw_idx, domain, label)
records = [("all", i, int(r.domain), int(r.label)) for i, r in stats_df.iterrows()]
ALL_RECORDS = records


class TerraRawSubset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[int(idx)]
        x = Image.open(row.filepath).convert("RGB")
        y = int(row.label)
        meta = np.array([int(row.domain)], dtype=np.int64)
        return x, y, meta


raw_subsets = {"all": TerraRawSubset(stats_df)}

print("\n========== TerraIncognita Dataset Stats ==========")
print("Total images:", len(stats_df))
print("Num classes:", NUM_CLASSES)
print("Classes:", [CLASS_NAME_MAP[i] for i in range(NUM_CLASSES)])
print("Num domains:", len(ALL_DOMAIN_IDS))
print("Domains:", [DOMAIN_NAME_MAP[d] for d in ALL_DOMAIN_IDS])

# Class distribution across domains.
counts = stats_df.groupby(["domain", "label"]).size().reset_index(name="count")

pivot = counts.pivot(index="domain", columns="label", values="count").fillna(0).astype(int)
pivot.index = [DOMAIN_NAME_MAP.get(int(d), str(d)) for d in pivot.index]
pivot.columns = [CLASS_NAME_MAP.get(int(c), str(c)) for c in pivot.columns]

plt.figure(figsize=(14, 6))
sns.heatmap(pivot, annot=True, fmt="d", cmap="YlGnBu", cbar_kws={"label": "Image Count"})
plt.title("TerraIncognita: Real Class Distribution Across Domains", fontsize=14, pad=12)
plt.xlabel("Class")
plt.ylabel("Domain")
plt.tight_layout()
plt.show()

global_counts = pivot.sum(axis=0).sort_values(ascending=False)
plt.figure(figsize=(10, 5))
global_counts.plot(kind="bar", color="teal")
plt.title("TerraIncognita: Global Class Distribution (Long-Tailed)", fontsize=14)
plt.ylabel("Total Number of Images")
plt.xlabel("Class")
plt.xticks(rotation=45)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

print("\n--- Global Imbalance Summary ---")
total_n = int(global_counts.sum())
for cls_name, count in global_counts.items():
    print(f"{cls_name:20}: {int(count):6d} images ({100.0*count/max(1,total_n):5.2f}%)")

print("\nReady for 4-fold domain holdout training.")

In [ ]:
# ==========================================
# TerraIncognita Data Pipeline + Model + Metrics
# ==========================================

from torchvision import transforms


def make_fraction_indices(n, fraction, seed_offset):
    if fraction >= 1.0:
        return np.arange(n, dtype=np.int64)
    keep = max(1, int(n * fraction))
    rng = np.random.RandomState(SEED + seed_offset)
    idx = rng.choice(n, size=keep, replace=False)
    idx.sort()
    return idx.astype(np.int64)


def to_pil(x):
    if isinstance(x, Image.Image):
        return x.convert("RGB")
    if torch.is_tensor(x):
        arr = x.detach().cpu()
        if arr.ndim == 3 and arr.shape[0] in (1, 3):
            arr = arr.permute(1, 2, 0).numpy()
        else:
            arr = arr.numpy()
        arr = np.clip(arr, 0, 255).astype(np.uint8)
        return Image.fromarray(arr).convert("RGB")
    if isinstance(x, np.ndarray):
        arr = np.clip(x, 0, 255).astype(np.uint8)
        return Image.fromarray(arr).convert("RGB")
    return x


image_size = SHARED_HPARAMS["image_size"]
train_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class TerraRecordDataset(Dataset):
    def __init__(self, records, raw_subsets, transform):
        self.records = records
        self.raw_subsets = raw_subsets
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        split_name, raw_idx, domain_id, _ = self.records[i]
        x, y, _ = self.raw_subsets[split_name][raw_idx]
        img = to_pil(x)
        img = self.transform(img)
        y_val = int(y.item()) if torch.is_tensor(y) else int(y)
        return img, y_val, int(i), int(domain_id)


def collate_fn(batch):
    xs, ys, idxs, groups = zip(*batch)
    x = torch.stack(xs, dim=0)
    y = torch.tensor(ys, dtype=torch.long)
    idx = torch.tensor(idxs, dtype=torch.long)
    g = torch.tensor(groups, dtype=torch.long)
    return x, y, idx, g


def split_records_for_holdout(all_records, test_domain_id, fraction):
    train_recs = [r for r in all_records if r[2] != test_domain_id]
    test_recs = [r for r in all_records if r[2] == test_domain_id]

    tr_idx = make_fraction_indices(len(train_recs), fraction, seed_offset=100 + int(test_domain_id))
    te_idx = make_fraction_indices(len(test_recs), fraction, seed_offset=200 + int(test_domain_id))

    train_recs = [train_recs[int(i)] for i in tr_idx]
    test_recs = [test_recs[int(i)] for i in te_idx]
    return train_recs, test_recs


def build_loaders_for_holdout(test_domain_id, batch_size, fraction):
    train_recs, test_recs = split_records_for_holdout(ALL_RECORDS, test_domain_id, fraction)

    train_ds = TerraRecordDataset(train_recs, raw_subsets, train_transform)
    test_ds = TerraRecordDataset(test_recs, raw_subsets, eval_transform)

    common = {
        "batch_size": batch_size,
        "num_workers": SHARED_HPARAMS["num_workers"],
        "pin_memory": False,
        "collate_fn": collate_fn,
    }

    train_loader = DataLoader(train_ds, shuffle=True, **common)
    test_loader = DataLoader(test_ds, shuffle=False, **common)

    return train_loader, test_loader, train_ds, test_ds


def to_device_batch(x, y, g=None):
    x = x.to(device, non_blocking=True)
    y = y.to(device, non_blocking=True)
    if g is None:
        return x, y
    g = g.to(device, non_blocking=True)
    return x, y, g


def fresh_model(num_classes):
    model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=num_classes)
    return model.to(device)


@torch.no_grad()
def eval_overall_accuracy(model, loader):
    model.eval()
    total_correct, total_n = 0, 0
    for x, y, _, _ in loader:
        x, y = to_device_batch(x, y)
        preds = model(x).argmax(1)
        total_correct += (preds == y).sum().item()
        total_n += y.size(0)
    return total_correct / max(1, total_n)


@torch.no_grad()
def eval_metrics(model, loader, domain_ids):
    model.eval()
    ce_sum, total_n = 0.0, 0
    domain_to_idx = {d: i for i, d in enumerate(domain_ids)}
    group_correct = torch.zeros(len(domain_ids))
    group_total = torch.zeros(len(domain_ids))
    ce_none = nn.CrossEntropyLoss(reduction="none")

    for x, y, _, g in loader:
        x, y, g = to_device_batch(x, y, g)
        logits = model(x)
        ce_vec = ce_none(logits, y)
        preds = logits.argmax(1)

        ce_sum += ce_vec.sum().item()
        total_n += y.size(0)

        correct = (preds == y)
        for dom in domain_ids:
            k = domain_to_idx[dom]
            mask = (g == dom)
            if mask.any():
                group_correct[k] += correct[mask].sum().item()
                group_total[k] += mask.sum().item()

    present = group_total > 0
    group_accs = torch.zeros(len(domain_ids))
    group_accs[present] = group_correct[present] / group_total[present]
    worst_group = group_accs[present].min().item() if present.any() else float("nan")

    return ce_sum / max(1, total_n), worst_group


def probe_max_batch_size(test_domain_id, start=16, max_try=512):
    print("\n=== Probing max batch size on holdout domain", DOMAIN_NAME_MAP.get(test_domain_id, test_domain_id), "===")

    tried = []
    best = None
    bs = start

    while bs <= max_try:
        try:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            model = fresh_model(NUM_CLASSES)
            opt = torch.optim.Adam(
                model.parameters(),
                lr=SHARED_HPARAMS["learning_rate"],
                weight_decay=SHARED_HPARAMS["weight_decay"],
            )
            ce = nn.CrossEntropyLoss()

            # Probe true memory usage at exactly `bs` samples.
            x = torch.randn(bs, 3, SHARED_HPARAMS["image_size"], SHARED_HPARAMS["image_size"], device=device)
            y = torch.randint(low=0, high=NUM_CLASSES, size=(bs,), device=device)

            model.train()
            opt.zero_grad(set_to_none=True)
            loss = ce(model(x), y)
            loss.backward()
            opt.step()

            tried.append((bs, "ok"))
            best = bs
            print(f"batch {bs}: ok")
            del model, opt, x, y, loss
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            bs *= 2

        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                tried.append((bs, "oom"))
                print(f"batch {bs}: OOM")
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                break
            raise

    if best is None:
        raise RuntimeError("No feasible batch size found. Reduce image size or start lower.")

    # Balanced safety margin: step down one level from max success when possible.
    chosen = max(start, best // 2) if best > start else best
    print("Probe attempts:", tried)
    print("Best successful:", best, "| Chosen balanced batch:", chosen)
    return chosen


In [ ]:
# ==========================================
# Selected Algorithms + Single Holdout Run (Best-Validation Checkpoint)
# ==========================================

import copy


class SAMAdam(torch.optim.Adam):
    def __init__(self, params, lr=1e-3, rho=0.05, **kwargs):
        if rho <= 0:
            raise ValueError(f"rho must be positive, got {rho}")
        self.rho = float(rho)
        super().__init__(params, lr=lr, **kwargs)

    @torch.no_grad()
    def _grad_norm(self):
        norms = []
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is not None:
                    norms.append(p.grad.detach().norm(2))
        if not norms:
            return torch.tensor(0.0, device=device)
        return torch.norm(torch.stack(norms), p=2)

    @torch.no_grad()
    def _epsilon(self, scale):
        eps = []
        for group in self.param_groups:
            e_group = []
            for p in group["params"]:
                if p.grad is None:
                    e_group.append(None)
                    continue
                e = p.grad * scale
                p.add_(e)
                e_group.append(e)
            eps.append(e_group)
        return eps

    @torch.no_grad()
    def _restore(self, eps):
        for group, e_group in zip(self.param_groups, eps):
            for p, e in zip(group["params"], e_group):
                if e is not None:
                    p.sub_(e)


@torch.no_grad()
def _kappa_for_h(F_vals, P_hat, tau_t, h, distance_scale=1.0, min_div=1e-2):
    log_base = torch.log(P_hat.clamp_min(1e-30))
    logits = log_base + (F_vals * h)
    logits = torch.clamp(logits, min=logits.max() - 40.0, max=logits.max() + 1e-6)
    P = torch.softmax(logits, dim=0)
    E_F = torch.dot(P, F_vals)
    num = E_F - tau_t
    Dkl = (P * (torch.log(P.clamp_min(1e-30)) - torch.log(P_hat.clamp_min(1e-30)))).sum()
    den = distance_scale * torch.max(Dkl, torch.tensor(0.0, device=device)) + min_div
    kappa = num / torch.max(den, torch.tensor(1e-20, device=device))
    return kappa, num, Dkl, P


@torch.no_grad()
def _fast_secant_search(F_vals, P_hat, tau_t, h_prev, dist_scale=1.0, min_div=1e-2, h_max=100.0, max_iter=2):
    h0 = torch.tensor(max(0.0, h_prev - 2.0), device=device)
    h1 = torch.tensor(max(0.1, h_prev + 2.0), device=device)
    h0 = torch.clamp(h0, 0.0, h_max)
    h1 = torch.clamp(h1, 0.0, h_max)

    k0, _, _, _ = _kappa_for_h(F_vals, P_hat, tau_t, h0, dist_scale, min_div)
    k1, _, _, _ = _kappa_for_h(F_vals, P_hat, tau_t, h1, dist_scale, min_div)

    for _ in range(max_iter):
        diff = h1 - h0
        slope = (k1 - k0) / (diff + 1e-8)
        h_new = torch.where(torch.abs(slope) > 1e-12, h1 - k1 / slope, h1 + 1.0)
        h_new = torch.max(h_new, torch.tensor(0.0, device=device))
        h_new = torch.clamp(h_new, 0.0, h_max)
        k_new, _, _, _ = _kappa_for_h(F_vals, P_hat, tau_t, h_new, dist_scale, min_div)
        h0, k0 = h1, k1
        h1, k1 = h_new, k_new

    _, num, Dkl, P = _kappa_for_h(F_vals, P_hat, tau_t, h1, dist_scale, min_div)
    return h1.item(), num, Dkl, P


def split_records_for_single_holdout(all_records, test_domain_id, fraction, val_fraction):
    train_pool = [r for r in all_records if r[2] != test_domain_id]
    test_recs = [r for r in all_records if r[2] == test_domain_id]

    tr_pool_idx = make_fraction_indices(len(train_pool), fraction, seed_offset=100 + int(test_domain_id))
    te_idx = make_fraction_indices(len(test_recs), fraction, seed_offset=200 + int(test_domain_id))

    train_pool = [train_pool[int(i)] for i in tr_pool_idx]
    test_recs = [test_recs[int(i)] for i in te_idx]

    if len(train_pool) < 2:
        raise ValueError("Not enough non-test training samples to make a train/val split.")

    if not (0.0 < float(val_fraction) < 1.0):
        raise ValueError(f"val_fraction must be in (0,1), got {val_fraction}")

    n_val = max(1, int(round(len(train_pool) * float(val_fraction))))
    n_val = min(n_val, len(train_pool) - 1)

    rng = np.random.RandomState(SEED + 300 + int(test_domain_id))
    perm = rng.permutation(len(train_pool))
    val_idx = set(perm[:n_val].tolist())

    val_recs = [rec for i, rec in enumerate(train_pool) if i in val_idx]
    train_recs = [rec for i, rec in enumerate(train_pool) if i not in val_idx]

    return train_recs, val_recs, test_recs


def build_loaders_for_single_holdout(test_domain_id, batch_size, fraction, val_fraction):
    train_recs, val_recs, test_recs = split_records_for_single_holdout(
        ALL_RECORDS,
        test_domain_id,
        fraction,
        val_fraction,
    )

    train_ds = TerraRecordDataset(train_recs, raw_subsets, train_transform)
    val_ds = TerraRecordDataset(val_recs, raw_subsets, eval_transform)
    test_ds = TerraRecordDataset(test_recs, raw_subsets, eval_transform)

    common = {
        "batch_size": batch_size,
        "num_workers": SHARED_HPARAMS["num_workers"],
        "pin_memory": False,
        "collate_fn": collate_fn,
    }

    train_loader = DataLoader(train_ds, shuffle=True, **common)
    val_loader = DataLoader(val_ds, shuffle=False, **common)
    test_loader = DataLoader(test_ds, shuffle=False, **common)

    return train_loader, val_loader, test_loader, train_ds, val_ds, test_ds


def resolve_test_domain_id(selector):
    if isinstance(selector, (int, np.integer)):
        sid = int(selector)
        if sid in ALL_DOMAIN_IDS:
            return sid
        raise ValueError(f"Unknown test domain id: {sid}. Available IDs: {ALL_DOMAIN_IDS}")

    if isinstance(selector, str):
        s = selector.strip()
        name_to_id = {name: did for did, name in DOMAIN_NAME_MAP.items()}
        if s in name_to_id:
            return name_to_id[s]

        if s.isdigit():
            sid = int(s)
            if sid in ALL_DOMAIN_IDS:
                return sid

        s_lower = s.lower()
        candidates = [did for did, name in DOMAIN_NAME_MAP.items() if s_lower in name.lower()]
        if len(candidates) == 1:
            return candidates[0]

    raise ValueError(
        "Unable to resolve TEST_DOMAIN_SELECTOR. "
        f"Got {selector}. Available domain names: {[DOMAIN_NAME_MAP[d] for d in ALL_DOMAIN_IDS]}"
    )


def train_erm(model, train_loader, val_loader, test_loader, *, epochs, lr, weight_decay, domain_ids):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    ce = nn.CrossEntropyLoss()

    tr_hist, val_hist = [], []
    te_o_hist, te_w_hist = [], []
    best_val, best_epoch = -1.0, -1
    best_val_test_overall = float("nan")
    best_val_test_worst = float("nan")
    best_state = None

    for ep in range(1, epochs + 1):
        model.train()
        total_loss, total_n = 0.0, 0

        for x, y, _, _ in tqdm(train_loader, leave=False, desc=f"ERM Ep {ep}"):
            x, y = to_device_batch(x, y)
            opt.zero_grad(set_to_none=True)
            logits = model(x)
            loss = ce(logits, y)
            loss.backward()
            opt.step()

            total_loss += loss.item() * y.size(0)
            total_n += y.size(0)

        tr_loss = total_loss / max(1, total_n)
        val_overall = eval_overall_accuracy(model, val_loader)
        _, test_worst = eval_metrics(model, test_loader, domain_ids=domain_ids)
        test_overall = eval_overall_accuracy(model, test_loader)

        tr_hist.append(tr_loss)
        val_hist.append(val_overall)
        te_o_hist.append(test_overall)
        te_w_hist.append(test_worst)

        if val_overall > best_val:
            best_val = val_overall
            best_epoch = ep
            best_val_test_overall = test_overall
            best_val_test_worst = test_worst
            best_state = copy.deepcopy(model.state_dict())

        print(
            f"[ERM] ep={ep:02d} tr={tr_loss:.4f} valOverall={val_overall:.4f} "
            f"testOverall={test_overall:.4f} testWorst={test_worst:.4f} bestVal={best_val:.4f}"
        )

    if globals().get("USE_BEST_VAL_CHECKPOINT", False) and (best_state is not None):
        model.load_state_dict(best_state)

    return {
        "train_hist": tr_hist,
        "val_overall_hist": val_hist,
        "test_overall_hist": te_o_hist,
        "test_worst_hist": te_w_hist,
        "best_epoch": best_epoch,
        "best_val_overall": best_val,
        "best_val_test_overall": best_val_test_overall,
        "best_val_test_worst": best_val_test_worst,
    }


def train_sam(model, train_loader, val_loader, test_loader, *, epochs, lr, weight_decay, rho, domain_ids):
    opt = SAMAdam(model.parameters(), lr=lr, weight_decay=weight_decay, rho=rho)
    ce = nn.CrossEntropyLoss()

    tr_hist, val_hist = [], []
    te_o_hist, te_w_hist = [], []
    best_val, best_epoch = -1.0, -1
    best_val_test_overall = float("nan")
    best_val_test_worst = float("nan")
    best_state = None

    for ep in range(1, epochs + 1):
        model.train()
        total_loss, total_n = 0.0, 0

        for step, (x, y, _, _) in enumerate(tqdm(train_loader, leave=False, desc=f"SAM Ep {ep}"), start=1):
            x, y = to_device_batch(x, y)

            opt.zero_grad(set_to_none=True)
            logits = model(x)
            loss = ce(logits, y)
            loss.backward()

            gnorm = opt._grad_norm()
            scale = rho / (gnorm + 1e-12)
            eps = opt._epsilon(scale)

            opt.zero_grad(set_to_none=True)
            logits_pert = model(x)
            loss_pert = ce(logits_pert, y)
            loss_pert.backward()

            opt._restore(eps)
            opt.step()

            total_loss += loss.item() * y.size(0)
            total_n += y.size(0)

            if step % 100 == 0:
                print(f"[SAM] ep={ep:02d} step={step:04d} loss={loss.item():.4f}")

        tr_loss = total_loss / max(1, total_n)
        val_overall = eval_overall_accuracy(model, val_loader)
        _, test_worst = eval_metrics(model, test_loader, domain_ids=domain_ids)
        test_overall = eval_overall_accuracy(model, test_loader)

        tr_hist.append(tr_loss)
        val_hist.append(val_overall)
        te_o_hist.append(test_overall)
        te_w_hist.append(test_worst)

        if val_overall > best_val:
            best_val = val_overall
            best_epoch = ep
            best_val_test_overall = test_overall
            best_val_test_worst = test_worst
            best_state = copy.deepcopy(model.state_dict())

        print(
            f"[SAM] ep={ep:02d} tr={tr_loss:.4f} valOverall={val_overall:.4f} "
            f"testOverall={test_overall:.4f} testWorst={test_worst:.4f} bestVal={best_val:.4f}"
        )

    if globals().get("USE_BEST_VAL_CHECKPOINT", False) and (best_state is not None):
        model.load_state_dict(best_state)

    return {
        "train_hist": tr_hist,
        "val_overall_hist": val_hist,
        "test_overall_hist": te_o_hist,
        "test_worst_hist": te_w_hist,
        "best_epoch": best_epoch,
        "best_val_overall": best_val,
        "best_val_test_overall": best_val_test_overall,
        "best_val_test_worst": best_val_test_worst,
    }


def train_irs_instance(model, train_loader, val_loader, test_loader, *, epochs, lr, weight_decay, domain_ids):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    ce_none = nn.CrossEntropyLoss(reduction="none")

    cfg = METHOD_HPARAMS["IRS-Instance"]
    tau = cfg["tau"]
    distance_scale = cfg["distance_scale"]
    min_div = cfg["min_div"]
    h_max = cfg["h_max"]
    tau_eps = cfg["tau_eps"]
    tau_mult = cfg["tau_mult"]
    warmup_epochs = cfg["warmup_epochs"]

    tr_hist, val_hist = [], []
    te_o_hist, te_w_hist = [], []
    best_val, best_epoch = float("-inf"), -1
    best_val_test_overall = float("nan")
    best_val_test_worst = float("nan")
    best_state = None

    prev_h = 0.0
    ce = nn.CrossEntropyLoss()

    for ep in range(1, epochs + 1):
        model.train()
        total_loss, total_n = 0.0, 0
        h_sum, h_cnt = 0.0, 0

        if ep <= warmup_epochs:
            for x, y, _, _ in tqdm(train_loader, leave=False, desc=f"IRS warmup Ep {ep}"):
                x, y = to_device_batch(x, y)
                opt.zero_grad(set_to_none=True)
                loss = ce(model(x), y)
                loss.backward()
                opt.step()
                total_loss += loss.item() * y.size(0)
                total_n += y.size(0)
        else:
            for x, y, _, _ in tqdm(train_loader, leave=False, desc=f"IRS Ep {ep}"):
                x, y = to_device_batch(x, y)
                opt.zero_grad(set_to_none=True)

                logits = model(x)
                ce_vec = ce_none(logits, y)

                F_b = ce_vec
                B = F_b.numel()
                if B <= 0:
                    continue

                P_b = torch.full_like(F_b, 1.0 / B)
                m_ref_b = torch.dot(F_b, P_b)
                maxF_b = F_b.max()

                tau_final_t = F_b.new_tensor(float(tau))
                tau_root = max(float(tau), float(m_ref_b.item()) * tau_mult)
                tau_root = min(tau_root, float(maxF_b.item()) - tau_eps)

                if tau_root <= float(m_ref_b.item()) + tau_eps:
                    continue

                tau_root_t = F_b.new_tensor(tau_root)

                try:
                    h_star, _, _, P_star_b = _fast_secant_search(
                        F_b,
                        P_b,
                        tau_root_t,
                        prev_h,
                        dist_scale=distance_scale,
                        min_div=min_div,
                        h_max=h_max,
                        max_iter=2,
                    )
                except Exception:
                    h_star = prev_h
                    _, _, _, P_star_b = _kappa_for_h(
                        F_b,
                        P_b,
                        tau_root_t,
                        F_b.new_tensor(h_star),
                        distance_scale,
                        min_div,
                    )

                r_wc = torch.dot(P_star_b, F_b)
                if r_wc <= tau_final_t + tau_eps:
                    continue

                loss = torch.dot(P_star_b.detach(), F_b)
                loss.backward()
                opt.step()

                total_loss += loss.item() * y.size(0)
                total_n += y.size(0)
                h_sum += min(h_star, h_max)
                h_cnt += 1
                prev_h = min(h_star, h_max)

        tr_loss = total_loss / max(1, total_n)
        val_overall = eval_overall_accuracy(model, val_loader)
        _, test_worst = eval_metrics(model, test_loader, domain_ids=domain_ids)
        test_overall = eval_overall_accuracy(model, test_loader)

        tr_hist.append(tr_loss)
        val_hist.append(val_overall)
        te_o_hist.append(test_overall)
        te_w_hist.append(test_worst)

        if ep > warmup_epochs and val_overall > best_val:
            best_val = val_overall
            best_epoch = ep
            best_val_test_overall = test_overall
            best_val_test_worst = test_worst
            best_state = copy.deepcopy(model.state_dict())

        best_val_str = f"{best_val:.4f}" if best_epoch > 0 else "NA"
        if ep <= warmup_epochs:
            print(
                f"[IRS warmup] ep={ep:02d} tr={tr_loss:.4f} valOverall={val_overall:.4f} "
                f"testOverall={test_overall:.4f} testWorst={test_worst:.4f} bestVal(non-warmup)={best_val_str}"
            )
        else:
            h_bar = h_sum / max(1, h_cnt)
            print(
                f"[IRS] ep={ep:02d} tr={tr_loss:.4f} valOverall={val_overall:.4f} "
                f"testOverall={test_overall:.4f} testWorst={test_worst:.4f} "
                f"bestVal(non-warmup)={best_val_str} h_bar={h_bar:.2f}"
            )

    if globals().get("USE_BEST_VAL_CHECKPOINT", False):
        if best_state is None:
            best_epoch = epochs
            best_val = val_hist[-1] if len(val_hist) > 0 else float("nan")
            best_val_test_overall = te_o_hist[-1] if len(te_o_hist) > 0 else float("nan")
            best_val_test_worst = te_w_hist[-1] if len(te_w_hist) > 0 else float("nan")
            best_state = copy.deepcopy(model.state_dict())
            print("[IRS] Warning: no non-warmup epoch was eligible for model selection; using final epoch model.")
        model.load_state_dict(best_state)

    return {
        "train_hist": tr_hist,
        "val_overall_hist": val_hist,
        "test_overall_hist": te_o_hist,
        "test_worst_hist": te_w_hist,
        "best_epoch": best_epoch,
        "best_val_overall": best_val,
        "best_val_test_overall": best_val_test_overall,
        "best_val_test_worst": best_val_test_worst,
    }


def cvar_loss_from_batch(loss_vec, alpha):
    alpha = max(min(float(alpha), 1.0), 1e-6)
    B = loss_vec.size(0)
    k = max(1, int(math.ceil(alpha * B)))
    topk_losses, _ = torch.topk(loss_vec, k, largest=True, sorted=False)
    return topk_losses.mean()


def train_cvar_dro(model, train_loader, val_loader, test_loader, *, epochs, lr, weight_decay, alpha, domain_ids):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    ce_none = nn.CrossEntropyLoss(reduction="none")

    tr_hist, val_hist = [], []
    te_o_hist, te_w_hist = [], []
    best_val, best_epoch = -1.0, -1
    best_val_test_overall = float("nan")
    best_val_test_worst = float("nan")
    best_state = None

    for ep in range(1, epochs + 1):
        model.train()
        total_loss, total_n = 0.0, 0

        for x, y, _, _ in tqdm(train_loader, leave=False, desc=f"CVaR Ep {ep}"):
            x, y = to_device_batch(x, y)
            opt.zero_grad(set_to_none=True)

            logits = model(x)
            ce_vec = ce_none(logits, y)
            loss = cvar_loss_from_batch(ce_vec, alpha)
            loss.backward()
            opt.step()

            total_loss += loss.item() * y.size(0)
            total_n += y.size(0)

        tr_loss = total_loss / max(1, total_n)
        val_overall = eval_overall_accuracy(model, val_loader)
        _, test_worst = eval_metrics(model, test_loader, domain_ids=domain_ids)
        test_overall = eval_overall_accuracy(model, test_loader)

        tr_hist.append(tr_loss)
        val_hist.append(val_overall)
        te_o_hist.append(test_overall)
        te_w_hist.append(test_worst)

        if val_overall > best_val:
            best_val = val_overall
            best_epoch = ep
            best_val_test_overall = test_overall
            best_val_test_worst = test_worst
            best_state = copy.deepcopy(model.state_dict())

        print(
            f"[CVaR] ep={ep:02d} tr={tr_loss:.4f} valOverall={val_overall:.4f} "
            f"testOverall={test_overall:.4f} testWorst={test_worst:.4f} bestVal={best_val:.4f}"
        )

    if globals().get("USE_BEST_VAL_CHECKPOINT", False) and (best_state is not None):
        model.load_state_dict(best_state)

    return {
        "train_hist": tr_hist,
        "val_overall_hist": val_hist,
        "test_overall_hist": te_o_hist,
        "test_worst_hist": te_w_hist,
        "best_epoch": best_epoch,
        "best_val_overall": best_val,
        "best_val_test_overall": best_val_test_overall,
        "best_val_test_worst": best_val_test_worst,
    }


@torch.no_grad()
def per_domain_accuracy(model, loader, domain_ids):
    model.eval()
    domain_to_idx = {d: i for i, d in enumerate(domain_ids)}
    group_correct = torch.zeros(len(domain_ids))
    group_total = torch.zeros(len(domain_ids))

    for x, y, _, g in loader:
        x, y, g = to_device_batch(x, y, g)
        preds = model(x).argmax(1)
        corr = (preds == y)
        for d in domain_ids:
            idx = domain_to_idx[d]
            mask = (g == d)
            if mask.any():
                group_correct[idx] += corr[mask].sum().item()
                group_total[idx] += mask.sum().item()

    out = {}
    for d in domain_ids:
        idx = domain_to_idx[d]
        out[d] = (group_correct[idx] / max(1.0, group_total[idx])).item()
    return out


@torch.no_grad()
def collect_test_outputs(model, loader):
    model.eval()

    sample_idx_all = []
    domain_all = []
    y_true_all = []
    y_pred_all = []
    logits_all = []
    probs_all = []

    for x, y, idx, g in loader:
        x, y, g = to_device_batch(x, y, g)
        logits = model(x)
        probs = torch.softmax(logits, dim=1)
        preds = logits.argmax(1)

        sample_idx_all.append(idx.cpu().numpy().astype(np.int64))
        domain_all.append(g.cpu().numpy().astype(np.int64))
        y_true_all.append(y.cpu().numpy().astype(np.int64))
        y_pred_all.append(preds.cpu().numpy().astype(np.int64))
        logits_all.append(logits.detach().cpu().numpy().astype(np.float32))
        probs_all.append(probs.detach().cpu().numpy().astype(np.float32))

    return {
        "sample_idx": np.concatenate(sample_idx_all, axis=0),
        "domain": np.concatenate(domain_all, axis=0),
        "y_true": np.concatenate(y_true_all, axis=0),
        "y_pred": np.concatenate(y_pred_all, axis=0),
        "logits": np.concatenate(logits_all, axis=0),
        "probs": np.concatenate(probs_all, axis=0),
    }


def _safe_name(text):
    return "".join(ch if ch.isalnum() or ch in ("-", "_") else "_" for ch in str(text))


def save_test_outputs_for_significance(method, holdout_name, outputs, test_ds, output_dir):
    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    method_tag = _safe_name(method)
    holdout_tag = _safe_name(holdout_name)
    base_name = f"{method_tag}_holdout_{holdout_tag}"

    sample_idx = outputs["sample_idx"].astype(np.int64)
    raw_idx = np.array([int(test_ds.records[int(i)][1]) for i in sample_idx], dtype=np.int64)
    max_prob = outputs["probs"].max(axis=1)

    csv_df = pd.DataFrame({
        "sample_idx": sample_idx,
        "raw_idx": raw_idx,
        "domain": outputs["domain"],
        "true_label": outputs["y_true"],
        "pred_label": outputs["y_pred"],
        "max_prob": max_prob,
        "filepath": [stats_df.iloc[int(r)].filepath for r in raw_idx],
    })

    csv_path = out_dir / f"{base_name}.csv"
    npz_path = out_dir / f"{base_name}.npz"

    csv_df.to_csv(csv_path, index=False)
    np.savez_compressed(
        npz_path,
        sample_idx=sample_idx,
        raw_idx=raw_idx,
        domain=outputs["domain"],
        true_label=outputs["y_true"],
        pred_label=outputs["y_pred"],
        logits=outputs["logits"],
        probs=outputs["probs"],
    )

    return str(csv_path), str(npz_path), int(len(csv_df))


# -------------------- Run config --------------------
METHOD_ENABLED = {
    "IRS-Instance": True,
    "SAM": True,
    "CVaR-DRO": True,
    "ERM": True,
}

TEST_DOMAIN_SELECTOR = "location_100"  # accepts domain id (int) or name (str)
VAL_FRACTION = 0.1
SKIP_IF_CHECKPOINT_EXISTS = False
USE_BEST_VAL_CHECKPOINT = False
SAVE_TEST_OUTPUTS = True
TEST_OUTPUTS_DIR = "significance_test_outputs"

RUN_METHODS = [m for m, enabled in METHOD_ENABLED.items() if enabled]
if len(RUN_METHODS) == 0:
    raise ValueError("No methods selected. Set at least one METHOD_ENABLED entry to True.")

set_seed(SEED)

results_by_fold = {}
summary_rows = []

selected_test_domain = resolve_test_domain_id(TEST_DOMAIN_SELECTOR)
selected_holdout_name = DOMAIN_NAME_MAP.get(selected_test_domain, str(selected_test_domain))

print("\n" + "=" * 80)
print("GLOBAL RUN CONFIG")
print("=" * 80)
print("Methods enabled:", RUN_METHODS)
print("Model:", MODEL_NAME)
print("Holdout selector:", TEST_DOMAIN_SELECTOR)
print("Resolved holdout domain:", selected_holdout_name, f"({selected_test_domain})")
print("DATA_FRACTION:", DATA_FRACTION)
print("VAL_FRACTION (random from pooled training records):", VAL_FRACTION)
print("Train batch size:", SHARED_HPARAMS["train_batch_size"])
print("Learning rate:", SHARED_HPARAMS["learning_rate"])
print("Num train epochs:", SHARED_HPARAMS["num_train_epochs"])
print("Weight decay:", SHARED_HPARAMS["weight_decay"])
print("Selection mode:", "best-val" if USE_BEST_VAL_CHECKPOINT else "final-epoch")
print("Save test outputs:", SAVE_TEST_OUTPUTS, "| dir:", TEST_OUTPUTS_DIR)
print("SAM rho:", METHOD_HPARAMS["SAM"]["rho"])
print("CVaR alpha:", METHOD_HPARAMS["CVaR-DRO"]["alpha"])
print("IRS params:", METHOD_HPARAMS["IRS-Instance"])
print("All domains:", [DOMAIN_NAME_MAP.get(d, d) for d in ALL_DOMAIN_IDS])
print("=" * 80)


test_domain = selected_test_domain
holdout_name = DOMAIN_NAME_MAP.get(test_domain, str(test_domain))
print("\n" + "#" * 80)
print(f"HOLDOUT DOMAIN: {holdout_name} ({test_domain})")
print("#" * 80)

train_loader, val_loader, test_loader, train_ds, val_ds, test_ds = build_loaders_for_single_holdout(
    test_domain_id=test_domain,
    batch_size=SHARED_HPARAMS["train_batch_size"],
    fraction=DATA_FRACTION,
    val_fraction=VAL_FRACTION,
)

print(f"train size: {len(train_ds)} | val size: {len(val_ds)} | test size: {len(test_ds)}")

fold_key = f"holdout_{test_domain}"
results_by_fold[fold_key] = {}

for method in RUN_METHODS:
    print("\n" + "-" * 80)
    print(f"Training {method} | holdout={holdout_name}")
    print("-" * 80)

    ckpt_name = f"ckpt_{method}_holdout_{test_domain}.pt"
    if SKIP_IF_CHECKPOINT_EXISTS and Path(ckpt_name).exists():
        print(f"Skipping {method} because checkpoint exists: {ckpt_name}")
        continue

    model = fresh_model(NUM_CLASSES)
    t0 = time.time()

    if method == "IRS-Instance":
        out = train_irs_instance(
            model,
            train_loader,
            val_loader,
            test_loader,
            epochs=SHARED_HPARAMS["num_train_epochs"],
            lr=SHARED_HPARAMS["learning_rate"],
            weight_decay=SHARED_HPARAMS["weight_decay"],
            domain_ids=ALL_DOMAIN_IDS,
        )
    elif method == "SAM":
        out = train_sam(
            model,
            train_loader,
            val_loader,
            test_loader,
            epochs=SHARED_HPARAMS["num_train_epochs"],
            lr=SHARED_HPARAMS["learning_rate"],
            weight_decay=SHARED_HPARAMS["weight_decay"],
            rho=METHOD_HPARAMS["SAM"]["rho"],
            domain_ids=ALL_DOMAIN_IDS,
        )
    elif method == "CVaR-DRO":
        out = train_cvar_dro(
            model,
            train_loader,
            val_loader,
            test_loader,
            epochs=SHARED_HPARAMS["num_train_epochs"],
            lr=SHARED_HPARAMS["learning_rate"],
            weight_decay=SHARED_HPARAMS["weight_decay"],
            alpha=METHOD_HPARAMS["CVaR-DRO"]["alpha"],
            domain_ids=ALL_DOMAIN_IDS,
        )
    elif method == "ERM":
        out = train_erm(
            model,
            train_loader,
            val_loader,
            test_loader,
            epochs=SHARED_HPARAMS["num_train_epochs"],
            lr=SHARED_HPARAMS["learning_rate"],
            weight_decay=SHARED_HPARAMS["weight_decay"],
            domain_ids=ALL_DOMAIN_IDS,
        )
    else:
        raise ValueError(f"Unknown method: {method}")

    elapsed = time.time() - t0

    val_overall_final = eval_overall_accuracy(model, val_loader)
    test_loss, test_worst = eval_metrics(model, test_loader, domain_ids=ALL_DOMAIN_IDS)
    test_overall = eval_overall_accuracy(model, test_loader)
    per_dom = per_domain_accuracy(model, test_loader, domain_ids=ALL_DOMAIN_IDS)

    output_csv_path, output_npz_path, n_output_rows = None, None, 0
    if SAVE_TEST_OUTPUTS:
        test_outputs = collect_test_outputs(model, test_loader)
        output_csv_path, output_npz_path, n_output_rows = save_test_outputs_for_significance(
            method=method,
            holdout_name=holdout_name,
            outputs=test_outputs,
            test_ds=test_ds,
            output_dir=TEST_OUTPUTS_DIR,
        )
        print(f"Saved test outputs: {output_csv_path} and {output_npz_path} ({n_output_rows} rows)")

    print(
        f"[{method}] final-epoch model -> valOverall={val_overall_final:.4f}, "
        f"testOverall={test_overall:.4f}, testWorst={test_worst:.4f}"
    )

    results_by_fold[fold_key][method] = {
        "train_hist": out["train_hist"],
        "val_overall_hist": out["val_overall_hist"],
        "test_overall_hist": out["test_overall_hist"],
        "test_worst_hist": out["test_worst_hist"],
        "best_epoch": out["best_epoch"],
        "best_val_overall": out["best_val_overall"],
        "best_val_test_overall": out["best_val_test_overall"],
        "best_val_test_worst": out["best_val_test_worst"],
        "final_val_overall": val_overall_final,
        "test_loss": test_loss,
        "worst_domain_acc": test_worst,
        "overall_acc": test_overall,
        "per_domain_acc": per_dom,
        "selection_mode": "best-val" if USE_BEST_VAL_CHECKPOINT else "final-epoch",
        "outputs_csv": output_csv_path,
        "outputs_npz": output_npz_path,
        "n_test_rows": n_output_rows,
        "time_sec": elapsed,
    }

    summary_rows.append({
        "holdout_domain": holdout_name,
        "method": method,
        "final_epoch": SHARED_HPARAMS["num_train_epochs"],
        "final_val_overall": val_overall_final,
        "test_overall_acc": test_overall,
        "test_worst_domain_acc": test_worst,
        "n_test_rows": n_output_rows,
        "time_min": elapsed / 60.0,
    })

    torch.save(
        {
            "method": method,
            "holdout_domain": int(test_domain),
            "model_state_dict": model.state_dict(),
            "selection": {
                "selection_mode": "best-val" if USE_BEST_VAL_CHECKPOINT else "final-epoch",
                "selected_epoch": out["best_epoch"] if USE_BEST_VAL_CHECKPOINT else SHARED_HPARAMS["num_train_epochs"],
                "best_epoch_by_val": out["best_epoch"],
                "best_val_overall": out["best_val_overall"],
                "final_val_overall": val_overall_final,
                "final_test_overall": test_overall,
                "final_test_worst": test_worst,
            },
            "config": {
                "model_name": MODEL_NAME,
                "shared_hparams": SHARED_HPARAMS,
                "method_hparams": METHOD_HPARAMS,
                "data_fraction": DATA_FRACTION,
                "val_fraction": VAL_FRACTION,
                "method_enabled": METHOD_ENABLED,
                "test_domain_selector": TEST_DOMAIN_SELECTOR,
                "use_best_val_checkpoint": USE_BEST_VAL_CHECKPOINT,
                "save_test_outputs": SAVE_TEST_OUTPUTS,
                "test_outputs_dir": TEST_OUTPUTS_DIR,
            },
        },
        ckpt_name,
    )

    print(f"Saved checkpoint: {ckpt_name}")

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

summary_df = pd.DataFrame(summary_rows)
print("\n" + "=" * 80)
print("Single-Holdout Results")
print("=" * 80)
if len(summary_df) == 0:
    print("No runs completed.")
else:
    print(summary_df)

if len(summary_df) > 0:
    rank_df = summary_df.sort_values("test_overall_acc", ascending=False).reset_index(drop=True)
    print("\nFinal Summary Table")
    print(f"{'Method':<15} | {'Final Ep':<8} | {'Final Val':<10} | {'Final Test':<11} | {'Final Worst':<11} | {'Rows':<6} | {'Time(min)':<9}")
    print("-" * 110)
    for _, r in rank_df.iterrows():
        print(
            f"{r['method']:<15} | {int(r['final_epoch']):>8d} | {100*r['final_val_overall']:>8.2f}% | "
            f"{100*r['test_overall_acc']:>9.2f}% | {100*r['test_worst_domain_acc']:>10.2f}% | "
            f"{int(r['n_test_rows']):>6d} | {r['time_min']:>8.2f}"
        )

print("\nTip: Set DATA_FRACTION=0.01 for smoke test, then DATA_FRACTION=1.0 for full run.")